In [1]:
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path, process_images, tokenizer_image_token
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN, IGNORE_INDEX
from llava.conversation import conv_templates, SeparatorStyle

from PIL import Image
import requests
import copy
import torch

import sys
import warnings

/home/gazran/miniforge3/envs/llava-next/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [2]:
# MODEL = "lmms-lab/llava-onevision-qwen2-0.5b-si"
MODEL = "lmms-lab/llava-onevision-qwen2-0.5b-ov"
# MODEL = "lmms-lab/llava-onevision-qwen2-7b-ov"
# MODEL = "lmms-lab/llava-onevision-qwen2-7b-si"
# MODEL = "lmms-lab/llava-onevision-qwen2-72b-si"

warnings.filterwarnings("ignore")
pretrained = MODEL
model_name = "llava_qwen"
conv_template = "qwen_2"  # Make sure you use correct chat template for different models
device = "cuda"
device_map = "auto"

In [3]:
tokenizer, model, image_processor, max_length = load_pretrained_model(pretrained, None, model_name, device_map=device_map)  # Add any other thing you want to pass in llava_model_args
model.eval();

Loaded LLaVA model: lmms-lab/llava-onevision-qwen2-0.5b-ov


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
You are using a model of type llava to instantiate a model of type llava_qwen. This is not supported for all configurations of models and can yield errors.


Loading vision tower: google/siglip-so400m-patch14-384
Model Class: LlavaQwenForCausalLM


In [4]:
image = Image.open('examples/groceries_world/cereal_in_container.png')
image2 = Image.open('examples/groceries_world/milk_in_container.png')
image_tensor = process_images([image, image2], image_processor, model.config)
image_tensor = [_image.to(dtype=torch.float16, device=device) for _image in image_tensor]

In [5]:
DOMAIN = """
(define (domain item-sorting)
  (:requirements :strips :typing :equality)
  (:types item container)
  (:predicates (on-table ?i - item)
	           (gripper-empty)
	           (gripping ?i - item)
               (in-container ?i - item ?c - container)
 )

  (:action pick-up
	     :parameters (?i - item)
	     :precondition (and (on-table ?i)(gripper-empty))
	     :effect
	     (and (not (on-table ?i))
		   (not (gripper-empty))
		   (gripping ?i)))

  (:action put-down
	     :parameters (?i - item)
	     :precondition (gripping ?i)
	     :effect
	     (and (not (gripping ?i))
		   (gripper-empty)
		   (on-table ?i)))

  (:action put-in-container
	     :parameters (?i - item ?c - container)
	     :precondition (and (gripping ?i))
	     :effect
	     (and (not (gripping ?i))
		   (gripper-empty)
		   (in-container ?i ?c)))

  (:action take-from-container
	     :parameters (?i - item ?c - container)
	     :precondition (and (in-container ?i ?c)(gripper-empty))
	     :effect
	     (and (gripping ?i)
		   (not (gripper-empty))
		   (not (in-container ?i ?c)))))
"""

PROBLEM = """
(define (problem grocery-bagging)

(:domain item-sorting)
(:objects
milk-carton lemon green-bottle loaf-of-bread red-box-of-cereal red-can-of-soda - item
dark-wood-container light-wood-container - container)

(:init

(gripper-empty)

(on-table milk-carton)
(on-table lemon)
(on-table green-bottle)
(on-table loaf-of-bread)
(on-table red-box-of-cereal)
(on-table red-can-of-soda)

)

(:goal (and
(in-container milk-carton dark-wood-container)
))

)
"""

all_queries = ['Is the milk carton on the table?',
 'Is the lemon on the table?',
 'Is the green bottle on the table?',
 'Is the loaf of bread on the table?',
 'Is the red box of cereal on the table?',
 'Is the red can of soda on the table?',
 'Is the gripper currently empty?',
 'Is the robot currently holding the milk carton?',
 'Is the robot currently holding the lemon?',
 'Is the robot currently holding the green bottle?',
 'Is the robot currently holding the loaf of bread?',
 'Is the robot currently holding the red box of cereal?',
 'Is the robot currently holding the red can of soda?',
 'Is the milk carton in the dark wood container?',
 'Is the milk carton inside the light wood container?',
 'Is the lemon in the dark-wood container?',
 'Is the lemon in the light-wood container?',
 'Is the green bottle in the dark wood container?',
 'Is the green bottle in the light wood container?',
 'Is the loaf of bread in the dark wood container?',
 'Is the loaf of bread in the light wood container?',
 'Is the red box of cereal in the dark wood container?',
 'Is the red box of cereal in the light wood container?',
 'Is the red can of soda inside the dark wood container?',
 'Is the red can of soda inside the light wood container?']

qs = '\n'.join(all_queries)

In [6]:
system = f"""The following are PDDL domain and problem files
{DOMAIN}
{PROBLEM}
Given a grounded predicate, you must determine whether it is true or false. Answer with a single "true" or "false" and no other text
"""
# system = f"""The following are questions of interest about a certain environment:
# {qs}
# Given an image of the environment and one of these questions, answer it truthfully with a single "yes" or "no" without any additional text.
# """

In [7]:
# query = "in-container(red-box-of-cereal, dark-wood-container)"
query = "what is in the images?"
question = DEFAULT_IMAGE_TOKEN + '\n' + query
conv = copy.deepcopy(conv_templates[conv_template])
conv.append_message(conv.roles[0], question)
conv.append_message(conv.roles[1], None)
# conv.system += """You will be asked a yes-no question about an image. Your job is to answer truthfully with a single "yes" or "no" and no other text."""
# conv.system += """You will be asked a yes-no question about an image. You must answer honestly and in detail, step by step. At the end of your response, provide your final answer in the format "final answer: <your answer>"."""

# conv.system += """You will be asked a series of yes-no question about an image. Repeat each question and answer it with a "yes" or "no" and no other text."""
# conv.system += system

prompt_question = conv.get_prompt()

In [9]:
input_ids = tokenizer_image_token(prompt_question, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").to(device)
image_sizes = [image.size, image2.size]

cont = model.generate(
    torch.stack([input_ids, input_ids]),
    images=image_tensor,
    image_sizes=image_sizes,
    do_sample=False,
    temperature=0,
    max_new_tokens=4096,
)
# text_outputs = tokenizer.batch_decode(cont, skip_special_tokens=True)
# print(text_outputs)

torch.Size([2, 27, 896])


../aten/src/ATen/native/cuda/Indexing.cu:1292: indexSelectLargeIndex: block: [289,0,0], thread: [0,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1292: indexSelectLargeIndex: block: [289,0,0], thread: [1,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1292: indexSelectLargeIndex: block: [289,0,0], thread: [2,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1292: indexSelectLargeIndex: block: [289,0,0], thread: [3,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1292: indexSelectLargeIndex: block: [289,0,0], thread: [4,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1292: indexSelectLargeIndex: block: [289,0,0], thread: [5,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1292: indexSelectLargeIndex: block: [289,0,0], 

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
type(model)

In [ ]:
cont.shape

In [ ]:
torch.stack([input_ids, input_ids]).shape

In [ ]:
image_tensor[0].shape